**Jakob should pip install "scikit-learn"**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from pathlib import Path
from datetime import datetime

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
file_path = DATA_DIR / "DMI_API_3stations.csv"

allPV_path = DATA_DIR / 'SyslabPV_15min_collective_PV.csv'
wind_path = DATA_DIR / 'SyslabWind_15min_nozeros.csv'
DMI_radiation_path = DATA_DIR / 'DMI_radiation.csv'
DMI_windspeed_path = DATA_DIR / 'DMI_ws.csv'

final_datapath = DATA_DIR / 'combined_data_DMI_gen.csv' #10 mins resolution

In [ ]:
#Create df
csv_pv = [allPV_path, DMI_radiation_path]
csv_wind = [wind_path, DMI_windspeed_path]

def model_df(file_paths):
    merged_df = None
    for path in file_paths:
        df = pd.read_csv(path,
                    delimiter=',',
                    decimal='.', 
                    parse_dates=['ts'],  # Your timestamp column name
                    index_col='ts')
        
        if merged_df is None:
            merged_df = df
        else:
            merged_df = pd.merge(merged_df, df, on="ts", how="outer")
    
    # Remove missing values because they're not useful for predictive model 
    merged_df = merged_df.dropna()

    return merged_df


In [ ]:
#Extract dataframes from csv files & create one for PV

locations = ['6156', '6174', '6188']

df_PV = model_df(csv_pv)

#Should I remove outliers? 
df_PV = df_PV[df_PV["Collective PV"] <= 50]

df_PV = df_PV.drop(columns=locations)

#Extract dataframes from csv files & create one for wind
df_wind = model_df([final_datapath])
    ##also account for wind power ~ v^3 (nonlinearity)

overlap_start = '2025-05-14 07:50:00'
overlap_end = '2025-12-31 23:50:00'

df_overlap_wind = df_wind.loc[overlap_start:overlap_end]

df_overlap_wind['Windspeed_cubed'] = df_overlap_wind['Windspeed']**3


In [ ]:
df_overlap_wind

In [ ]:
# Manipulate wind power to better assess cut-in speed

df_subwind = df_wind[['Windspeed', 'Aircon_WT Power']].copy()

# Create bins (e.g., 0.5 m/s intervals)
df_subwind['wind_bin'] = pd.cut(df_subwind['Windspeed'], bins=np.arange(0, df_subwind['Windspeed'].max()+0.5, 0.5))

# Average power per bin
binned_df = (df_subwind.groupby('wind_bin').agg({'Windspeed': 'mean','Aircon_WT Power': 'mean'}).reset_index())

# Plot binned power
binned_df['wind_mid'] = binned_df['wind_bin'].apply(lambda x: x.mid)
plt.xlabel('Binned Windspeeds')
plt.ylabel('Binned Wind Power')
plt.grid()
# Add footnote
plt.figtext(
    0.5, -0.05,
    'Note: Data binned in 0.5 m/s intervals. Cut-in estimated visually to be ~1 m/s and rated-power to be ~11 m/s.',
    ha='center',
    fontsize=9
)
plt.plot(binned_df['wind_mid'], binned_df['Aircon_WT Power'])

In [ ]:
# Assessing relationship between irradiance and PV powere
plt.figure(figsize=(12, 4))
plt.scatter(df_PV['mean radiation'], df_PV['Collective PV'], label='Measured PV power')
plt.xlabel("Irradiance")
plt.ylabel("Collective PV power")
plt.title("PV model: assess linearity with power data")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#Create train data with 80/20 data split for PV
df_train_PV = df_PV.sort_index().reset_index()

# Removing very low-radiation values => helps visualization of model but it drops the performance
#df_train_PV = df_train_PV[df_train_PV['mean radiation'] > 0.1]

split_index = int(len(df_train_PV)*0.8)

train_PV = df_train_PV.iloc[:split_index] #everything up to 80%
test_PV = df_train_PV.iloc[split_index:] #everything after 80%

# Fit model PV based on the early 80%
X_trainPV = train_PV[['mean radiation']] 
y_trainPV = train_PV['Collective PV']

X_testPV = test_PV[['mean radiation']]
y_testPV = test_PV['Collective PV']

pv_model = LinearRegression()
pv_model.fit(X_trainPV, y_trainPV)

# Predict for all weather timestamps
y_predPV = pv_model.predict(X_testPV)

# Evaluate model (on test set)
pv_r2 = r2_score(y_testPV, y_predPV)
pv_mae = mean_absolute_error(y_testPV, y_predPV)
pv_rmse = mean_squared_error(y_testPV, y_predPV) ** 0.5

print("PV model")
print("Slope:", pv_model.coef_[0])
print("Intercept:", pv_model.intercept_)
print("R²:", pv_r2)
print("MAE:", pv_mae)
print("RMSE:", pv_rmse)

# Predict for all weather timestamps
df_PV['pv_pred'] = pv_model.predict(df_PV[['mean radiation']])

# Prevent negative values
df_PV['pv_pred'] = df_PV['pv_pred'].clip(lower=0)

The current version (above) removed the outlier of PV production, which output ~50 kW

Removing no radiation <= best to not remove any radiation
- PV model
- Slope: 0.0633294165254058
- Intercept: -0.18760957284146507
- R²: 0.6224439821264665
- MAE: 1.089923544791629
- RMSE: 3.376467260529961

Removing radiation <0.1
- Slope: 0.06379791823954216
- Intercept: -0.6578908421551724
- R²: 0.6116664135792969
- MAE: 3.42203285383305
- RMSE: 6.246333198336148

Removing radiation <1
- Slope: 0.06383264299805723
- Intercept: -0.7339483290252851
- R²: 0.6070169804664096
- MAE: 3.7021677151249985
- RMSE: 6.6124003636982955


Removing radiation <10
- PV model
- Slope: 0.06424293511992434
- Intercept: -0.9337705295494807
- R²: 0.5632482117730956
- MAE: 4.14605146609768
- RMSE: 7.0714386655282

In [ ]:
# Scatter plot: measured vs predicted on test set
plt.figure(figsize=(6, 6))
plt.scatter(y_testPV, y_predPV, alpha=0.6)
plt.xlabel("Measured PV power")
plt.ylabel("Predicted PV power")
plt.title("PV model: measured vs predicted (test set)")

min_val = min(y_testPV.min(), y_predPV.min())
max_val = max(y_testPV.max(), y_predPV.max())
plt.plot([min_val, max_val], [min_val, max_val])

plt.show()

# Time series plot: test period only
plt.figure(figsize=(12, 4))
plt.plot(test_PV['ts'], y_testPV, label='Measured PV power')
plt.plot(test_PV['ts'], y_predPV, label='Predicted PV power')
plt.xlabel("Time")
plt.ylabel("PV power")
plt.title("PV model: measured and predicted over time (test set)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#Create train data with 80/20 data split for wind
df_train_wind = df_overlap_wind.sort_index().reset_index()
split_index = int(len(df_train_wind)*0.8)

train_wind = df_train_wind.iloc[:split_index] #everything up to 80%
test_wind = df_train_wind.iloc[split_index:] #everything after 80%

# Fit model PV based on the early 80%
X_trainwind = train_wind[['Windspeed', 'Windspeed_cubed']] 
y_trainwind = train_wind['Aircon_WT Power']

X_testwind = test_wind[['Windspeed', 'Windspeed_cubed']]
y_testwind = test_wind['Aircon_WT Power']

wind_model = LinearRegression()
wind_model.fit(X_trainwind, y_trainwind)

# Predict for all weather timestamps
y_predwind = wind_model.predict(X_testwind)

#Make 0 all production happening below cut-in wind speed
cut_in_speed = 1  # what is the turbine's value? check wiki!
rated_power = 8

y_predwind = pd.Series(y_predwind, index=test_wind.index)

y_predwind[test_wind['Windspeed'] < cut_in_speed] = 0
y_predwind = y_predwind.clip(lower=0, upper= rated_power)

# Evaluate model (on test set)
wind_r2 = r2_score(y_testwind, y_predwind)
wind_mae = mean_absolute_error(y_testwind, y_predwind)
wind_rmse = mean_squared_error(y_testwind, y_predwind) ** 0.5

print("Wind model")
print("Slope:", wind_model.coef_[0])
print("Intercept:", wind_model.intercept_)
print("R²:", wind_r2)
print("MAE:", wind_mae)
print("RMSE:", wind_rmse)

# Predict for all weather timestamps
df_overlap_wind['wind_pred'] = wind_model.predict(df_overlap_wind[['Windspeed', 'Windspeed_cubed']])

#Make 0 those production values happening below the cut-in speed & prevent negative values
df_overlap_wind.loc[df_overlap_wind['Windspeed'] < cut_in_speed, 'wind_pred'] = 0
df_overlap_wind['wind_pred'] = df_overlap_wind['wind_pred'].clip(lower=0)


In [ ]:
# Scatter plot: measured vs predicted on test set
plt.figure(figsize=(6, 6))
plt.scatter(y_testwind, y_predwind, alpha=0.6)
plt.xlabel("Measured Wind power")
plt.ylabel("Predicted Wind power")
plt.title("Wind model: measured vs predicted (test set)")

min_val = min(y_testwind.min(), y_predwind.min())
max_val = max(y_testwind.max(), y_predwind.max())
plt.plot([min_val, max_val], [min_val, max_val])

plt.show()

# Time series plot: test period only
plt.figure(figsize=(12, 4))
plt.plot(test_wind['ts'], y_testwind, label='Measured Wind power')
plt.plot(test_wind['ts'], y_predwind, label='Predicted Wind power')
plt.xlabel("Time")
plt.ylabel("Wind power")
plt.title("Wind model: measured and predicted over time (test set)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Lasso Regression with Lags

In [ ]:
# --- Add lag feature BEFORE splitting ---
df_overlap_wind = df_overlap_wind.sort_index()

df_overlap_wind['wind_lag_1'] = df_overlap_wind['Windspeed'].shift(1)
df_overlap_wind['wind_lag_2'] = df_overlap_wind['Windspeed'].shift(2)
df_overlap_wind['wind_lag_3'] = df_overlap_wind['Windspeed'].shift(3)
df_overlap_wind['wind_lag_4'] = df_overlap_wind['Windspeed'].shift(4)
df_overlap_wind['wind_lag_5'] = df_overlap_wind['Windspeed'].shift(5)

# Drop NaNs caused by lag
df_overlap_wind = df_overlap_wind.dropna()

# --- Prepare data ---
df_train_wind = df_overlap_wind.reset_index()
df_train_wind['ts'] = pd.to_datetime(df_train_wind['ts'])

split_index = int(len(df_train_wind) * 0.8)
train_wind = df_train_wind.iloc[:split_index]
test_wind = df_train_wind.iloc[split_index:]

# --- Features ---
features = ['Windspeed', 'Windspeed_cubed', 'wind_lag_1', 'wind_lag_2', 'wind_lag_3', 'wind_lag_4', 'wind_lag_5']

X_train = train_wind[features]
y_train = train_wind['Aircon_WT Power']

X_test = test_wind[features]
y_test = test_wind['Aircon_WT Power']

# --- Try different alpha values ---
'''
Wind model with lag - alpha comparison
 alpha     R²    MAE   RMSE
0.0001 0.5808 0.7674 1.0903
0.0010 0.5811 0.7672 1.0900
0.0100 0.5806 0.7683 1.0906
0.1000 0.5709 0.7846 1.1032
1.0000 0.3290 1.0498 1.3795
'''

alphas = [0.0010]
results = []

for a in alphas:
    model = make_pipeline(
        StandardScaler(),
        Lasso(alpha=a)
    )

    model.fit(X_train, y_train)

    y_pred = pd.Series(model.predict(X_test), index=test_wind.index)

    # Apply physical constraints
    y_pred[test_wind['Windspeed'] < cut_in_speed] = 0
    y_pred = y_pred.clip(lower=0, upper=rated_power)

    # Metrics
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results.append({
        'alpha': a,
        'R²': r2,
        'MAE': mae,
        'RMSE': rmse
    })

# --- Print results nicely ---
results_df = pd.DataFrame(results)
print("Wind model with lag - alpha comparison")
print(results_df.to_string(index=False, float_format="%.4f".__mod__))

Wind model with lag - alpha comparison
 alpha     R²    MAE   RMSE
0.0001 0.5808 0.7674 1.0903
0.0010 0.5811 0.7672 1.0900
0.0100 0.5806 0.7683 1.0906
0.1000 0.5709 0.7846 1.1032
1.0000 0.3290 1.0498 1.3795

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.5)

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val])

plt.xlabel("Measured")
plt.ylabel("Predicted")
plt.title("Measured vs Predicted")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(test_wind['ts'], y_test, label='Measured')
plt.plot(test_wind['ts'], y_pred, label='Predicted')
plt.xlabel("Time")
plt.ylabel("Power")
plt.title("Wind Power (Test Set)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()